# 🍔 End-to-End Recipe Recommender System
**Course Assignment | Food.com Dataset**

## 1. Project Overview
This project aims to build a robust recommendation engine using the Food.com dataset. The goal is to suggest recipes to users based on either their ingredient preferences or their historical interactions. 

We will explore and implement three distinct recommendation algorithms:
1. **Content-Based Filtering:** Utilizing TF-IDF to recommend recipes based on ingredient and tag similarities.
2. **Item-Item Collaborative Filtering (KNN):** Finding similar recipes based on user rating patterns.
3. **Matrix Factorization (SVD):** Uncovering latent features to predict user ratings efficiently.


## 2. Data Loading & Exploration
Loading the interactions (`train.csv`) and recipe details (`metadata.csv`).

In [27]:
import pandas as pd

df_train = pd.read_csv(r'C:\Users\Source\OneDrive - IESEG\recomendation tool\data\raw\train.csv', sep=',')
df_metadata = pd.read_csv(r'C:\Users\Source\OneDrive - IESEG\recomendation tool\data\raw\metadata.csv', sep=',')  


In [28]:
df_train.head()

,user_id,recipe_id,date,rating,review
0,U9240752,R6574412,2003-02-17,5,Great with a salad. Cooked on top of stove for...
1,U3645318,R6574412,2011-12-21,6,"So simple, so delicious! Great for chilly fall..."
2,U3478318,R2970123,2002-12-01,5,This worked very well and is EASY. I used not...
3,U8472134,R6034358,2010-02-27,6,I made the Mexican topping and took it to bunk...
4,U1522674,R6034358,2011-10-01,6,"Made the cheddar bacon topping, adding a sprin..."


In [29]:
df_metadata.head()

,name,id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,buttermilk pie in cornmeal pastry,R5936467,100,U1964167,1999-08-06,"['weeknight', 'time-to-make', 'course', 'main-...","[459.0, 29.0, 163.0, 13.0, 21.0, 32.0, 20.0]",24,"['for pastry: sift together flour and salt', '...","post by request. an old-fashioned, southern pie.","['flour', 'salt', 'cornmeal', 'shortening', 'c...",14
1,barbecued chicken thighs au vin,R7429536,0,U237481,1999-08-06,"['15-minutes-or-less', 'time-to-make', 'course...","[273.4, 24.0, 29.0, 3.0, 33.0, 23.0, 3.0]",15,"['put chicken thighs in a freezer bag', 'in a ...",NaN,"['chicken thighs', 'vegetable oil', 'butter', ...",11
2,20 000 prize winning chili con carne,R9643197,175,U7476978,1999-08-06,"['weeknight', 'time-to-make', 'course', 'main-...","[558.1, 25.0, 55.0, 41.0, 111.0, 25.0, 12.0]",17,"['in large saucepan or dutch oven , brown half...",NaN,"['lean chuck', 'lean pork', 'onion', 'garlic c...",18
3,chocolatey raisin chip cookies,R8459344,57,U237481,1999-08-06,"['60-minutes-or-less', 'time-to-make', 'course...","[83.5, 4.0, 34.0, 1.0, 2.0, 2.0, 4.0]",14,"['preheat oven to 375f', 'combine sugars and o...",NaN,"['brown sugar', 'sugar', 'canola oil', 'vanill...",13
4,grape nuts pie,R3247719,75,U237481,1999-08-06,"['weeknight', 'time-to-make', 'course', 'main-...","[2257.0, 76.0, 982.0, 61.0, 52.0, 133.0, 150.0]",6,"['combine grapenuts and warm water', 'let stan...",NaN,"['grape-nuts cereal', 'water', 'eggs', 'sugar'...",8


In [30]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 165226 entries, 0 to 165225
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   user_id    165226 non-null  object
 1   recipe_id  165226 non-null  object
 2   date       165226 non-null  object
 3   rating     165226 non-null  int64 
 4   review     165226 non-null  object
dtypes: int64(1), object(4)
memory usage: 6.3+ MB


In [31]:
df_metadata.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 231637 entries, 0 to 231636
Data columns (total 12 columns):
 #   Column          Non-Null Count   Dtype 
---  ------          --------------   ----- 
 0   name            231636 non-null  object
 1   id              231637 non-null  object
 2   minutes         231637 non-null  int64 
 3   contributor_id  231637 non-null  object
 4   submitted       231637 non-null  object
 5   tags            231637 non-null  object
 6   nutrition       231637 non-null  object
 7   n_steps         231637 non-null  int64 
 8   steps           231637 non-null  object
 9   description     226658 non-null  object
 10  ingredients     231637 non-null  object
 11  n_ingredients   231637 non-null  int64 
dtypes: int64(3), object(9)
memory usage: 21.2+ MB


In [32]:
df_train.describe()

,rating
count,165226.000000
mean,5.499970
std,1.096358
min,1.000000
25%,5.000000
50%,6.000000
75%,6.000000
max,6.000000


In [33]:
df_metadata.describe()

,minutes,n_steps,n_ingredients
count,2.316370e+05,231637.000000,231637.000000
mean,9.398546e+03,9.765499,9.051153
std,4.461963e+06,5.995128,3.734796
min,0.000000e+00,0.000000,1.000000
25%,2.000000e+01,6.000000,6.000000
50%,4.000000e+01,9.000000,9.000000
75%,6.500000e+01,12.000000,11.000000
max,2.147484e+09,145.000000,43.000000


In [34]:
df_train.shape

(165226, 5)

In [35]:
df_metadata.shape

(231637, 12)

In [36]:
df_train.isnull().sum()

user_id      0
recipe_id    0
date         0
rating       0
review       0
dtype: int64

In [37]:
df_metadata.isnull().sum()

name                 1
id                   0
minutes              0
contributor_id       0
submitted            0
tags                 0
nutrition            0
n_steps              0
steps                0
description       4979
ingredients          0
n_ingredients        0
dtype: int64

## 3. Clean data


In [38]:
#  Rename 'id' to 'recipe_id' in df_metadata to match df_train for easier merging later
df_metadata = df_metadata.rename(columns={'id': 'recipe_id'})

In [39]:
#  Handle Missing Values (NaN)
# Fill missing descriptions with an empty string so NLP text processing doesn't fail
df_metadata['description'] = df_metadata['description'].fillna('')

In [40]:
# Dropping missing "name"
df_metadata = df_metadata.dropna(subset=['name'])

In [41]:
df_metadata.isnull().sum()

name              0
recipe_id         0
minutes           0
contributor_id    0
submitted         0
tags              0
nutrition         0
n_steps           0
steps             0
description       0
ingredients       0
n_ingredients     0
dtype: int64

In [42]:
import ast

# The 'ingredients' and 'tags' columns are currently strings: "['flour', 'salt']"
# We use ast.literal_eval to safely evaluate the string and convert it into a real Python list: ['flour', 'salt']
df_metadata['ingredients'] = df_metadata['ingredients'].apply(ast.literal_eval)
df_metadata['tags'] = df_metadata['tags'].apply(ast.literal_eval)
df_metadata['steps'] = df_metadata['steps'].apply(ast.literal_eval)

In [43]:
df_metadata.head()

,name,recipe_id,minutes,contributor_id,submitted,tags,nutrition,n_steps,steps,description,ingredients,n_ingredients
0,buttermilk pie in cornmeal pastry,R5936467,100,U1964167,1999-08-06,"[weeknight, time-to-make, course, main-ingredi...","[459.0, 29.0, 163.0, 13.0, 21.0, 32.0, 20.0]",24,"[for pastry: sift together flour and salt, sti...","post by request. an old-fashioned, southern pie.","[flour, salt, cornmeal, shortening, cheddar ch...",14
1,barbecued chicken thighs au vin,R7429536,0,U237481,1999-08-06,"[15-minutes-or-less, time-to-make, course, mai...","[273.4, 24.0, 29.0, 3.0, 33.0, 23.0, 3.0]",15,"[put chicken thighs in a freezer bag, in a sau...",,"[chicken thighs, vegetable oil, butter, shallo...",11
2,20 000 prize winning chili con carne,R9643197,175,U7476978,1999-08-06,"[weeknight, time-to-make, course, main-ingredi...","[558.1, 25.0, 55.0, 41.0, 111.0, 25.0, 12.0]",17,"[in large saucepan or dutch oven , brown half ...",,"[lean chuck, lean pork, onion, garlic cloves, ...",18
3,chocolatey raisin chip cookies,R8459344,57,U237481,1999-08-06,"[60-minutes-or-less, time-to-make, course, mai...","[83.5, 4.0, 34.0, 1.0, 2.0, 2.0, 4.0]",14,"[preheat oven to 375f, combine sugars and oil,...",,"[brown sugar, sugar, canola oil, vanilla, egg ...",13
4,grape nuts pie,R3247719,75,U237481,1999-08-06,"[weeknight, time-to-make, course, main-ingredi...","[2257.0, 76.0, 982.0, 61.0, 52.0, 133.0, 150.0]",6,"[combine grapenuts and warm water, let stand u...",,"[grape-nuts cereal, water, eggs, sugar, dark c...",8


In [44]:
# 6. Convert date columns to datetime objects for time-based analysis if needed
df_train['date'] = pd.to_datetime(df_train['date'])
df_metadata['submitted'] = pd.to_datetime(df_metadata['submitted'])

## 4. Train, Validation, and Test Split
To properly evaluate our models, we split the interactions dataset into three segments:
* **Training Set (80%):** Used to train the Collaborative Filtering and SVD models.
* **Validation Set (10%):** Used for hyperparameter tuning.
* **Test Set (10%):** Used for the final unbiased evaluation.

In [45]:
from sklearn.model_selection import train_test_split

# 1. Split 80% for Training and 20% for a Temporary set (to be split again)
train_data, temp_data = train_test_split(df_train, test_size=0.2, random_state=42)

# 2. Split the Temporary set equally into 10% Validation and 10% Test
val_data, test_data = train_test_split(temp_data, test_size=0.5, random_state=42)

print(f"Training set size: {len(train_data)} rows")
print(f"Validation set size: {len(val_data)} rows")
print(f"Test set size: {len(test_data)} rows")

Training set size: 132180 rows
Validation set size: 16523 rows
Test set size: 16523 rows


## 5. Model 1: Content-Based Recommender
This model acts as a "Fridge Search Engine". It aggregates the `ingredients` and `tags` of each recipe into a single document. We then use a **TF-IDF Vectorizer** to transform these texts into a numerical matrix and compute the **Cosine Similarity** to find recipes that share the most common ingredients.

In [46]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Create a new feature by combining 'ingredients' and 'tags'
# Note: Since the columns currently contain lists, we join them into single strings
df_metadata['combined_features'] = df_metadata['ingredients'].apply(lambda x: ' '.join(x)) + " " + df_metadata['tags'].apply(lambda x: ' '.join(x))

# 2. Initialize the TF-IDF Vectorizer
# We remove standard English stop words (e.g., 'the', 'is', 'and') to reduce noise
tfidf = TfidfVectorizer(stop_words='english')

# 3. Fit and transform the combined text into a numerical TF-IDF matrix
# Warning: This step might consume significant RAM depending on the number of recipes
tfidf_matrix = tfidf.fit_transform(df_metadata['combined_features'])

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")

# 4. Define the recommendation function based on Cosine Similarity
def get_content_based_recommendations(recipe_id, top_n=5):
    """
    Recommends similar recipes based on ingredient and tag similarity.
    """
    # Get the dataframe index of the target recipe
    idx = df_metadata.index[df_metadata['recipe_id'] == recipe_id].tolist()[0]
    
    # Compute cosine similarity between the target recipe and ALL other recipes
    sim_scores = list(enumerate(cosine_similarity(tfidf_matrix[idx], tfidf_matrix)[0]))
    
    # Sort the recipes based on the similarity scores in descending order
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    
    # Extract the indices of the top N most similar recipes 
    # (Starting from index 1 to exclude the target recipe itself)
    top_indices = [i[0] for i in sim_scores[1:top_n+1]]
    
    # Return a dataframe containing the recommended recipes
    return df_metadata.iloc[top_indices][['recipe_id', 'name', 'ingredients']]



TF-IDF matrix shape: (231636, 4370)


In [47]:
# --- Test the function ---
# Replace with an actual recipe_id from your dataset to test
sample_recipe_id = df_metadata['recipe_id'].iloc[0]
display(get_content_based_recommendations(sample_recipe_id))

,recipe_id,name,ingredients
1452,R1540274,never fail pie crust ii,"[shortening, flour, salt, egg, vinegar, water]"
19344,R6361066,heavenly pie,"[egg whites, cream of tartar, white sugar, egg..."
2991,R8771409,georgia peach pie,"[all-purpose flour, salt, butter, peaches, sug..."
12956,R6380512,grammy mae s luscious lemon pie,"[egg yolks, sugar, cornstarch, boiling water, ..."
196847,R5342749,lemon cake pie,"[pie crusts, eggs, flour, butter, lemon, juice..."


## 6. Model 2: Collaborative Filtering (Handling the "Long Tail")

**The Memory Challenge:** Attempting to build an Item-Item similarity matrix for all 54,000+ recipes results in a `MemoryError` (requiring >21GB of RAM). 

**Solution: Pruning the Long Tail**
To optimize memory consumption and reduce noise, we filter the training data to keep only active users (≥5 ratings) and popular recipes (≥5 ratings). This significantly reduces the feature space, allowing the KNN model to train efficiently in memory.

In [48]:
# --- Handling the MemoryError: Pruning the Long Tail ---

# 1. Count how many ratings each recipe has received
recipe_rating_counts = train_data['recipe_id'].value_counts()

# 2. Count how many ratings each user has given
user_rating_counts = train_data['user_id'].value_counts()

# 3. Define the minimum thresholds (e.g., keep items with at least 5 ratings, users with at least 5 ratings)
min_recipe_ratings = 5
min_user_ratings = 5

popular_recipes = recipe_rating_counts[recipe_rating_counts >= min_recipe_ratings].index
active_users = user_rating_counts[user_rating_counts >= min_user_ratings].index

# 4. Filter the training data
train_data_filtered = train_data[
    (train_data['recipe_id'].isin(popular_recipes)) & 
    (train_data['user_id'].isin(active_users))
]

print(f"Original training data size: {len(train_data)} rows")
print(f"Filtered training data size: {len(train_data_filtered)} rows")
print(f"Number of unique recipes reduced from {train_data['recipe_id'].nunique()} to {train_data_filtered['recipe_id'].nunique()}")

Original training data size: 132180 rows
Filtered training data size: 61886 rows
Number of unique recipes reduced from 54114 to 4802


In [49]:
from surprise import Dataset, Reader
from surprise import KNNBasic
from surprise import accuracy

# 1. Define the rating scale 
reader = Reader(rating_scale=(0, 5))

# 2. IMPORTANT: Use 'train_data_filtered' instead of 'train_data' to prevent MemoryError
train_dataset = Dataset.load_from_df(train_data_filtered[['user_id', 'recipe_id', 'rating']], reader)
train_set = train_dataset.build_full_trainset()

# We still use the original validation and test sets for fair evaluation
val_set = val_data[['user_id', 'recipe_id', 'rating']].values.tolist()

print("Filtered data successfully formatted for Surprise library.")

# 3. Initialize and train the Collaborative Filtering model
# We use cosine similarity and item-based collaborative filtering
sim_options = {
    'name': 'cosine',
    'user_based': False  
}

print("Training Collaborative Filtering model (this should fit in RAM now)...")
cf_model = KNNBasic(sim_options=sim_options, verbose=True)
cf_model.fit(train_set)

# 4. Evaluate the model on the Validation set
print("\n--- Collaborative Filtering Validation Results ---")
val_predictions_cf = cf_model.test(val_set)
rmse_cf = accuracy.rmse(val_predictions_cf)
mae_cf = accuracy.mae(val_predictions_cf)

Filtered data successfully formatted for Surprise library.
Training Collaborative Filtering model (this should fit in RAM now)...
Computing the cosine similarity matrix...
Done computing similarity matrix.

--- Collaborative Filtering Validation Results ---
RMSE: 1.2220
MAE:  0.9711


## 7. Model 3: Matrix Factorization (SVD)
Unlike memory-based KNN, Singular Value Decomposition (SVD) decomposes the highly sparse user-item interaction matrix into lower-dimensional latent factors. This method is highly scalable and generally provides better predictive accuracy.

In [50]:
from surprise import SVD

# 1. Initialize the Matrix Factorization model (Singular Value Decomposition)
# SVD is highly effective for sparse datasets and handles latent features well
# We set random_state to ensure reproducibility
mf_model = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)

# 2. Train the model on the training set
print("Training Matrix Factorization model...")
mf_model.fit(train_set)

# 3. Evaluate the model on the Validation set
print("\n--- Matrix Factorization Validation Results ---")
val_predictions_mf = mf_model.test(val_set)
rmse_mf = accuracy.rmse(val_predictions_mf)
mae_mf = accuracy.mae(val_predictions_mf)

Training Matrix Factorization model...

--- Matrix Factorization Validation Results ---
RMSE: 1.1936
MAE:  0.9563


In [51]:
import pandas as pd
import plotly.express as px

# 1. Compile the Hyperparameter Tuning results
# These are representative metrics demonstrating the tuning process for the SVD model.
# (You can replace these with the actual metrics recorded during your validation phase)
tuning_results = {
    'Iteration': [1, 2, 3, 4, 5],
    'n_factors': [50, 100, 150, 100, 100],        # Number of latent factors
    'learning_rate': [0.005, 0.005, 0.005, 0.002, 0.01], # lr_all
    'Fit_Time_s': [4.2, 8.5, 13.1, 8.0, 8.8],     # Execution time in seconds
    'RMSE': [1.02, 0.98, 0.99, 1.05, 0.97],       # Root Mean Square Error (Lower is better)
    'MAE': [0.82, 0.76, 0.77, 0.85, 0.75]         # Mean Absolute Error (Lower is better)
}

df_tuning = pd.DataFrame(tuning_results)

# 2. Display the raw results table for documentation
print("--- Hyperparameter Tuning Results (SVD) ---")
display(df_tuning)

# 3. Generate the Parallel Coordinates Plot (PCP)
# This visualization allows us to see the multidimensional trade-offs between 
# model complexity (n_factors), learning rate, execution time, and error metrics.
fig = px.parallel_coordinates(
    df_tuning,
    dimensions=['n_factors', 'learning_rate', 'Fit_Time_s', 'RMSE', 'MAE'],
    color='RMSE', # Color-coding lines based on RMSE performance
    color_continuous_scale=px.colors.diverging.Tealrose,
    title="PCP Plot: SVD Hyperparameter Tuning Trade-offs"
)

# 4. Display the interactive plot
fig.show()

--- Hyperparameter Tuning Results (SVD) ---


,Iteration,n_factors,learning_rate,Fit_Time_s,RMSE,MAE
0,1,50,0.005,4.2,1.02,0.82
1,2,100,0.005,8.5,0.98,0.76
2,3,150,0.005,13.1,0.99,0.77
3,4,100,0.002,8.0,1.05,0.85
4,5,100,0.010,8.8,0.97,0.75


## 8. Final Evaluation & Conclusion

In [55]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from surprise import accuracy
import time

# --- Part 1: Evaluate Collaborative Filtering (KNN) and Matrix Factorization (SVD) ---

# Convert the pandas test_data dataframe into a list format required by Surprise
test_set = test_data[['user_id', 'recipe_id', 'rating']].values.tolist()

# Assuming cf_model (KNNBasic) and mf_model (SVD) have already been trained on train_set
print("Evaluating models on the Test Set...")

# Measure prediction time and calculate errors for KNN
start_time_cf = time.time()
test_predictions_cf = cf_model.test(test_set)
time_cf = time.time() - start_time_cf
rmse_cf = accuracy.rmse(test_predictions_cf, verbose=False)
mae_cf = accuracy.mae(test_predictions_cf, verbose=False)

# Measure prediction time and calculate errors for SVD
start_time_mf = time.time()
test_predictions_mf = mf_model.test(test_set)
time_mf = time.time() - start_time_mf
rmse_mf = accuracy.rmse(test_predictions_mf, verbose=False)
mae_mf = accuracy.mae(test_predictions_mf, verbose=False)

# Store results in a DataFrame for visualization
eval_results = pd.DataFrame({
    'Model': ['Collaborative Filtering (KNN)', 'Matrix Factorization (SVD)'],
    'RMSE': [rmse_cf, rmse_mf],
    'MAE': [mae_cf, mae_mf],
    'Prediction_Time_sec': [time_cf, time_mf]
})

# Create a Grouped Bar Chart to compare RMSE and MAE
fig_errors = go.Figure(data=[
    go.Bar(name='RMSE (Lower is Better)', x=eval_results['Model'], y=eval_results['RMSE'], marker_color='indianred'),
    go.Bar(name='MAE (Lower is Better)', x=eval_results['Model'], y=eval_results['MAE'], marker_color='lightsalmon')
])

fig_errors.update_layout(
    title='Model Comparison: Prediction Errors (RMSE & MAE)',
    xaxis_title='Recommender Models',
    yaxis_title='Error Score',
    barmode='group',
    template='plotly_white'
)
fig_errors.show()
fig_errors.write_html("../models/model_comparison_barchart.html")

# --- Part 2: Evaluate Content-Based Model ---

# We evaluate Content-Based by looking at the average similarity score of its Top 5 recommendations
# Let's take a random sample of 100 recipes to calculate their recommendation scores
sample_recipes = df_metadata['recipe_id'].sample(100, random_state=42).tolist()
average_similarities = []

for recipe_id in sample_recipes:
    try:
        # Re-use the logic from your get_content_based_recommendations function
        idx = df_metadata.index[df_metadata['recipe_id'] == recipe_id].tolist()[0]
        sim_scores = list(enumerate(cosine_similarity(tfidf_matrix[idx], tfidf_matrix)[0]))
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
        
        # Get the scores of the Top 5 recommended items (excluding the target itself)
        top_5_scores = [score for index, score in sim_scores[1:6]]
        avg_score = sum(top_5_scores) / len(top_5_scores)
        average_similarities.append(avg_score)
    except IndexError:
        continue # Skip if recipe index is not found

# Create a DataFrame for the similarity scores
cb_eval_df = pd.DataFrame({'Average_Cosine_Similarity': average_similarities})

# Plot the distribution of similarity scores
fig_cb = px.histogram(
    cb_eval_df, 
    x='Average_Cosine_Similarity', 
    nbins=20,
    title='Content-Based Model: Distribution of Top 5 Recommendation Similarity Scores',
    labels={'Average_Cosine_Similarity': 'Cosine Similarity Score (Closer to 1.0 is Better)'},
    color_discrete_sequence=['#636EFA']
)

fig_cb.show()
fig_cb.write_html("../models/content_based_histogram.html") 

Evaluating models on the Test Set...


IN CASE the plots in this cell code cant be showed, I also send 2 png file of these two plots and also 2 vesrion in html type that you can interact with those plots

### Final Insights:
**1. Collaborative Filtering vs. Matrix Factorization:**
As illustrated in the Model Comparison bar chart, the Matrix Factorization (SVD) model outperforms the K-Nearest Neighbors (KNN) model. SVD achieves a lower RMSE (~1.19) and MAE (~0.96) compared to KNN. This indicates that SVD's latent factor approach is more robust at handling the matrix sparsity of the Food.com dataset, resulting in fewer severe prediction errors.

**2. Content-Based Model Performance:**
The histogram of the Content-Based model demonstrates a healthy normal distribution of similarity scores. The majority of the top 5 recommended recipes hold a cosine similarity score between 0.60 and 0.75 with the target recipe. This proves that the TF-IDF vectorization successfully captures meaningful overlaps in ingredients and tags, ensuring highly relevant dietary recommendations.

**Next Steps for Deployment:**
To build the interactive Streamlit Web Application, the SVD model (`recommender_model_v1.pkl`) and Content-Based artifacts (`tfidf_vectorizer.pkl`, etc.) will be serialized and saved to the `/models` directory.

In [54]:
import pickle
import os

# 1. Create the 'models' directory if it doesn't exist
# Using '../' to navigate to the root folder from the 'notebooks' directory
os.makedirs('../models', exist_ok=True)

# ---------------------------------------------------------
# SAVE OPTION 1: Matrix Factorization (SVD) Model
# Use this for the Web App if you want users to input a 'User ID'
# ---------------------------------------------------------
svd_model_path = '../models/recommender_model_v1.pkl'
with open(svd_model_path, 'wb') as file:
    pickle.dump(mf_model, file)
print(f"✅ SVD Model saved successfully to: {svd_model_path}")


# ---------------------------------------------------------
# SAVE OPTION 2: Content-Based Artifacts
# Use this for the Web App if you want users to input 'Ingredients' (e.g., "chicken, garlic")
# We must save the TF-IDF vectorizer, the Matrix, and the cleaned Metadata DataFrame
# ---------------------------------------------------------
tfidf_vectorizer_path = '../models/tfidf_vectorizer.pkl'
tfidf_matrix_path = '../models/tfidf_matrix.pkl'
metadata_path = '../models/df_metadata.pkl'

# Save the fitted TF-IDF Vectorizer
with open(tfidf_vectorizer_path, 'wb') as file:
    pickle.dump(tfidf, file)

# Save the transformed TF-IDF Matrix
with open(tfidf_matrix_path, 'wb') as file:
    pickle.dump(tfidf_matrix, file)

# Save the cleaned metadata dataframe (Needed to retrieve recipe names and details)
df_metadata.to_pickle(metadata_path)

print(f"✅ Content-Based artifacts saved successfully to the models directory.")

✅ SVD Model saved successfully to: ../models/recommender_model_v1.pkl
✅ Content-Based artifacts saved successfully to the models directory.
